# 주주총회 소집공고 테스트

OpenDART API를 통해 주주총회 소집공고를 가져오고, 의안별 내용을 확인하는 테스트 노트북.

In [ ]:
import asyncio
import sys
import re
sys.stdout.reconfigure(encoding='utf-8')

from open_proxy_mcp.dart.client import DartClient

client = DartClient()
print('DartClient 초기화 완료')

## 1. 소집공고 검색

종목코드, 회사명, 또는 corp_code로 검색 가능.

In [ ]:
# === 여기에 검색할 기업 입력 ===
QUERY = "033780"  # 종목코드, 회사명, 또는 corp_code
# ================================

async def search_agm(query: str):
    # 기업 조회
    corp = await client.lookup_corp_code(query)
    if not corp:
        print(f"'{query}'에 해당하는 기업을 찾을 수 없습니다.")
        print("종목코드(예: 033780)로 다시 시도해보세요.")
        return None, None
    
    print(f"회사명: {corp['corp_name']}")
    print(f"종목코드: {corp['stock_code']}")
    print(f"corp_code: {corp['corp_code']}")
    print()
    
    # 소집공고 검색
    result = await client.search_filings_by_ticker(
        ticker=query,
        bgn_de="20260101",
        end_de="20261231",
        pblntf_ty="E",
    )
    
    filings = [
        item for item in result.get("list", [])
        if "소집" in item.get("report_nm", "")
    ]
    
    if not filings:
        print("주주총회 소집공고가 없습니다.")
        return corp, None
    
    print(f"소집공고 {len(filings)}건 발견:\n")
    for i, f in enumerate(filings):
        print(f"  [{i}] {f['report_nm']} | {f['rcept_dt']} | {f['rcept_no']}")
    
    return corp, filings

corp, filings = await search_agm(QUERY)

## 2. 소집공고 본문 가져오기

위 결과에서 원하는 공고의 인덱스를 선택.

In [ ]:
# === 위 결과에서 인덱스 선택 ===
FILING_INDEX = 0
# ================================

async def fetch_agm_document(filings, index: int):
    if not filings or index >= len(filings):
        print("유효한 인덱스를 선택하세요.")
        return None, None
    
    target = filings[index]
    print(f"가져오는 중: {target['report_nm']} ({target['rcept_no']})\n")
    
    doc = await client.get_document(target["rcept_no"])
    text = doc["text"]
    images = doc["images"]
    
    print(f"본문 길이: {len(text):,}자")
    if images:
        print(f"첨부 이미지: {images}")
    
    # 의안 목록 파싱
    agenda_items = parse_agenda(text)
    
    print(f"\n{'='*60}")
    print(f"의안 {len(agenda_items)}개 발견:\n")
    for i, item in enumerate(agenda_items):
        print(f"  [{i}] {item['title']}")
    
    return text, agenda_items

def parse_agenda(text: str) -> list[dict]:
    """본문에서 의안 목록 파싱"""
    items = []
    
    # 기본 정보 (일시, 장소)
    date_match = re.search(r'일\s*시\s*[:：]?\s*(.+?)(?=\d+\.\s*장|$)', text)
    place_match = re.search(r'장\s*소\s*[:：]?\s*(.+?)(?=\d+\.\s*회|$)', text)
    
    if date_match or place_match:
        basic_info = "주총 기본정보"
        if date_match:
            basic_info += f" | 일시: {date_match.group(1).strip()[:50]}"
        if place_match:
            basic_info += f" | 장소: {place_match.group(1).strip()[:80]}"
        items.append({"title": basic_info, "keyword": "일시"})
    
    # 제N호 의안 패턴 추출
    pattern = r'제(\d+[-\d]*)호[^:：]*[:：]?\s*(.+?)(?=제\d+[-\d]*호|□|■|$)'
    matches = re.findall(pattern, text[:5000])  # 앞부분 의안 목록에서만
    
    seen = set()
    for num, title in matches:
        title_clean = title.strip()[:100]
        key = f"제{num}호"
        if key not in seen and title_clean:
            seen.add(key)
            items.append({"title": f"{key}: {title_clean}", "keyword": key})
    
    return items

full_text, agenda_items = await fetch_agm_document(filings, FILING_INDEX)

## 3. 의안 상세 보기

위 의안 목록에서 인덱스를 선택하면 해당 내용을 표시.

In [ ]:
# === 위 의안 목록에서 인덱스 선택 ===
AGENDA_INDEX = 0
# ====================================

def show_agenda_detail(text: str, agenda_items: list[dict], index: int, context_chars: int = 3000):
    if not agenda_items or index >= len(agenda_items):
        print("유효한 인덱스를 선택하세요.")
        return
    
    item = agenda_items[index]
    keyword = item["keyword"]
    
    print(f"{'='*60}")
    print(f"  {item['title']}")
    print(f"{'='*60}\n")
    
    # 경영참고사항 섹션에서 해당 의안 찾기 (뒷부분에 상세 내용)
    # □ 또는 ■ + 제N호 패턴으로 찾기
    detail_pattern = f'[□■].*?{re.escape(keyword)}'
    matches = list(re.finditer(detail_pattern, text))
    
    if matches:
        # 마지막 매치가 보통 상세 내용 (앞부분은 목록)
        start = matches[-1].start()
        excerpt = text[start:start + context_chars]
        print(excerpt)
        if start + context_chars < len(text):
            print(f"\n... (더 보려면 context_chars를 늘리세요)")
    else:
        # 키워드로 직접 검색
        idx = text.find(keyword)
        if idx >= 0:
            # 좀 더 뒤에서 찾기 (상세 내용은 보통 뒷부분)
            idx2 = text.find(keyword, idx + len(keyword))
            pos = idx2 if idx2 >= 0 else idx
            excerpt = text[max(0, pos - 100):pos + context_chars]
            print(excerpt)
        else:
            print("해당 의안의 상세 내용을 찾을 수 없습니다.")

show_agenda_detail(full_text, agenda_items, AGENDA_INDEX)

## 4. 전체 본문 (필요 시)

전체 본문이 필요하면 아래 셀 실행.

In [ ]:
# 전체 본문 출력 (길 수 있음)
# print(full_text)